## Configuration

In [ ]:
from pathlib import Path
import tomllib
from utils.utils import *
import json

PATH_EEG = Path("../EEG/EEG/results/")
FILENAME = 'Case_03_ExG.csv'
MARKER_FILE = 'Case_03_markers.csv'
CONFIG_PATH = "../EEG/EEG/config/config.toml"

with open(CONFIG_PATH, "rb") as f:
    config = tomllib.load(f)

BANDS = config['BANDS']
SEGMENT_DEF = config['SEGMENT_DEF']
RANDOM_SEED = config['RANDOM_SEED']
DURATION = config['DURATION']

SFREQ = config['SFREQ']
L_FREQ = config['L_FREQ']
H_FREQ = config['H_FREQ']

## Raw Data

In [ ]:
raw_full, df_full, df_markers_full = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)


raws, labels = extract_segments(raw_full, df_full, df_markers_full, SEGMENT_DEF)

In [ ]:
plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha', bands=BANDS)
plot_topomap_comparison(raws, labels, band_name='Beta', bands=BANDS)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta', bands=BANDS)
plot_topomap_comparison(raws, labels, band_name='Theta', bands=BANDS)

## Preprocessing

In [ ]:
import mne
from mne.preprocessing import ICA
raw, df, df_markers = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

# ICA
# n_components=15 (16 Channels, ICA finding max n_channels - 1 or same number)
ica = ICA(n_components=15, max_iter='auto', random_state=RANDOM_SEED)

print(">>> ICA (Might take a while)...")
ica.fit(raw)

print(">>> Showing Components...")
ica.plot_components()
plt.show()

ica.plot_sources(raw, show_scrollbars=True)
plt.show()

In [ ]:
exclude_indices = [0, 1, 2, 5, 7, 9] 

print(f">>> Removing ingredients: {exclude_indices}")
ica.exclude = exclude_indices

raw_clean = raw.copy()
ica.apply(raw_clean)

print(">>> Cleaned data.")

target_chan = 'F3' if 'F3' in raw.ch_names else raw.ch_names[0]

print(f"Comparison on the channel {target_chan}...")
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(raw.get_data(picks=target_chan)[0, 1000:2500], color='red', alpha=0.5, label='Before cleaning (Original)')
ax.plot(raw_clean.get_data(picks=target_chan)[0, 1000:2500], color='black', label='After cleaning (Clean)')
ax.set_title(f"Artifact Removal Effect (Eyes and Muscles) - Channel {target_chan}")
ax.legend()
plt.show()

In [ ]:
raws, labels = extract_segments(raw_clean, df_full, df_markers_full, SEGMENT_DEF)

plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha', bands=BANDS)
plot_topomap_comparison(raws, labels, band_name='Beta', bands=BANDS)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta', bands=BANDS)
plot_topomap_comparison(raws, labels, band_name='Gamma', bands=BANDS)

#### (Beta/Gamma)
W fazach wymagających skupienia (Smart i Smart Scroll) nie ma drastycznego wygaszenia prawej kory wzrokowej. Obie półkule potyliczne (O1 i O2) na mapach Beta i Gamma dają równomierne sygnały. Z kolei faza Brainrot (lewa góra, Gamma) wywołuje masywny pożar zdominowany silniej po lewej stronie.

Uczestnik nie filtruje obrazu na rzecz samego tekstu w trybie Smart – przetwarza wymagające treści bardzo zbalansowanie, korzystając z obu półkul (łączy tekst z graficznym kontekstem). To "Brainrot" zmusza lewą (analityczną) półkulę do wysiłku wizualnego.

#### (Delta/Gamma)
W Fazie 1 odnotowano całkowity brak Delty z przodu głowy oraz specyficzną, silną aktywność Gamma w lewym pasie czołowo-skroniowo-centralnym ("Czerwona Wyspa" po lewej stronie mapy Gamma).

Brainrot nie wprowadza tego uczestnika w stan otępienia ("Zombie Mode"). Przeciwnie – wywołuje reakcję aktywnej oceny. Obecność silnej fali Gamma w lewym obszarze (odpowiedzialnym za przetwarzanie języka i logikę) przy jednoczesnej czujności czołowej sugeruje występowanie aktywnego, krytycznego "komentarza wewnętrznego" w reakcji na chaotyczne treści. Uczestnik nie  chłonie tego biernie, ale to ocenia.

#### (Delta/Alpha)
W Fazie 3 (Brainrot Scroll) lewa kora wzrokowa (O1) zostaje nagle i drastycznie wyłączona – na mapie Delta pojawia się ciemnoczerwona plama skrajnego znużenia, a na mapie Alpha równie czerwona plama uśpienia/braku uwagi. Reszta mózgu (w tym kora ruchowa i czołowa) pozostaje w pełni wybudzona (granatowa w Delcie).

W trybie szybkiego scrollowania "głupot" mózg jest tak przebodźcowany, że celowo, awaryjnie odcina zmysł wzroku (O1 wchodzi w stan głębokiego uśpienia/zmęczenia). Scrollowanie nie jest tu kierowane tym, co widzą oczy. Decyzja o ruchu palcem zapada "w ciemno", na poziomie czołowo-motorycznym. Palec ucieka przed bodźcem, zanim oko zdąży go przetworzyć.

## Spektogram

In [ ]:
# Alpha (fatigue/relax)
plot_band_power_over_time(raws, labels, band=(8, 12), band_name='Alpha')
# Beta (Focus)
plot_band_power_over_time(raws, labels, band=(13, 30), band_name='Beta')

In [ ]:
# Emotion
calculate_asymmetry(raws, labels, ch_left='F3', ch_right='F4')

# According to Heller's model, parietal asymmetry is responsible for physiological arousal
calculate_asymmetry(raws, labels, ch_left='P3', ch_right='P4')

# Image processing method.
calculate_asymmetry(raws, labels, ch_left='O1', ch_right='O2')

# Motor
calculate_asymmetry(raws, labels, ch_left='C3', ch_right='C4')

### Podsumowanie

**Brainrot**: 
Brainrot to dla badanego sensoryczny atak. Kora wzrokowa wykonuje pracę, próbując zdekodować obrazy. Ponieważ jednocześnie na czole (F4 vs F3) jest wyraźny minus, oznacza to, że mózg jest tym przebodźcowany i bardzo mu się to nie podoba. Brak ruchu i pozornie "niskie emocje" mogą wynikać z oszołomienia, a nie z relaksu.

**Scrollowanie**:
Na wykresie P4 vs P3 widać, że słupki są dodatnie, ale drastycznie spadają (zbliżają się do zera) w momentach wejścia w tryb Scroll (zarówno Brainrot Scroll, jak i Smart Scroll). Mniejszy plus oznacza, że do głosu dochodzi prawa kora ciemieniowa (P4). A ona włącza się w stanach dezorientacji, lęku przestrzennego i wysokiego pobudzenia (arousal). Zatem scroll działa jak "strzał kofeiny", ale uderza w układ stresu przestrzennego.

**Smart Scroll**:
Wskaźnik czołowy (F4 vs F3) sięga tu najgłębszego dna z całego badania. Połączenie konieczności czytania mądrych treści, praworęcznej nawigacji (wysokie C4 vs C3) oraz stresu wynikającego ze zmieniającego się obrazu sprawia, że płaty czołowe krzyczą, żeby to skończyć. To może być najtrudniejsze i najbardziej nieprzyjemne zadanie poznawcze dla tego badanego.

## Scroll

In [ ]:
PROJECT_ROOT = Path.cwd().parent 

USER_MAP_PATH = PROJECT_ROOT / "EEG" / "EEG" / "user_map.json"
INTERACTIONS_PATH = PROJECT_ROOT / "EEG" / "research_events_rows.csv"
USERS_PATH = PROJECT_ROOT / "EEG" / "Users_rows.csv"

FILE_NAME = 'Case_03_ExG.csv'

In [ ]:
scroll_data = extract_scroll_events(FILE_NAME, INTERACTIONS_PATH, USERS_PATH, USER_MAP_PATH)

if scroll_data is not None and not scroll_data.empty:
    new_annotations = mne.Annotations(
        onset=scroll_data['onset'].values,
        duration=scroll_data['duration'].values,
        description=scroll_data['description'].values
    )

    raw_clean.set_annotations(new_annotations)
    
    print("SUCCESS: Annotations applied to raw_clean.")
    print(f"Time range of scrolls: {scroll_data['onset'].min():.2f}s to {scroll_data['onset'].max():.2f}s")
else:
    print("WARNING: No scroll data found. Plots will be empty.")

In [ ]:
compare_stages(
    raw_clean, 
    band='Beta', 
    fmin=13, 
    fmax=30, 
    channel='C3',  
    duration=DURATION
)

In [ ]:
# REWARD/MOTIVATION SYSTEM (Frontal Alpha)
# Channel: F3 (Left Frontal)
# Band: Alpha (8-12 Hz)
# Interpretation: THE LOWER THE GRAPH, THE GREATER THE “REWARD/ENGAGEMENT.”

compare_stages(
    raw_clean, 
    band='Alpha', 
    fmin=8, 
    fmax=12, 
    channel='F3',   # Key channel for positive/aspirational emotions
    duration=DURATION
)

In [ ]:
# ISUAL RELAXATION (Alpha)
# Channel: O1 (Back of the head)
# Band: Alpha (8-12 Hz)
# Interpretation: HIGH LINE = RELAXATION / ZOMBIE MODE. LOW = WORK.

compare_stages(
    raw_clean, 
    band='Alpha', 
    fmin=8, 
    fmax=12, 
    channel='O1', 
    duration=DURATION
)

In [ ]:
# INTELLECTUAL EFFORT (Frontal Theta) ---
# Channel: Fz (Center of forehead)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = STRONG THINKING/MEMORIZATION.

target_channel = 'Fz'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# COGNITIVE CONTROL & INHIBITION (Right Frontal Theta) ---
# Channel: F4 (Right forehead / Right Frontal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = SUSTAINED FOCUS & SUPPRESSING DISTRACTIONS (Mental effort to stay on task or regulate emotional impulses).

target_channel = 'F4'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# VERBAL & ANALYTICAL MEMORY (Left Parietal Theta) ---
# Channel: P3 (Left back of head / Left Parietal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = PROCESSING TEXT/LANGUAGE (Reading captions, understanding speech, verbal working memory).

target_channel = 'P3'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# VISUOSPATIAL PROCESSING & ATTENTION (Right Parietal Theta) ---
# Channel: P4 (Right back of head / Right Parietal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = ENCODING VISUAL/SPATIAL INFO (Processing layout, movement, spatial memory, or complex imagery).

target_channel = 'P4'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

#### Kora wzrokowa (Alpha O1) 

**Brainrot Scroll**: Tuż po uderzeniu palcem w ekran, zamiast fizjologicznego spadku, można zaobserwować nagłe, wystrzały mocy fali Alpha (synchronizacja). To oznacza, że scroll działa tu jak fizyczny włącznik "ślepoty" – mózg na ułamek sekundy odcina zmysł wzroku.

**Smart Scroll**: Przy przewijaniu mądrych treści, fale Alpha zachowują się nieco spokojniej, próbując wrócić do analitycznego czytania (lekkie spadki po scrollu), ale sygnał wciąż jest wyraźnie poszarpany.

#### Płaty ciemieniowe (Theta P3, P4)

**Brainrot Scroll**: Na prawej korze ciemieniowej (P4) po każdej czerwonej linii – występują ostre, bardzo wyraźne piki fali Theta. W tym samym czasie lewa półkula (P3) reaguje znacznie słabiej. Ten asymetryczny wystrzał na P4 to fizjologiczna sygnatura nagłego pobudzenia i stresu przestrzennego wywołanego zmianą bodźca.

**Smart Scroll**: Podobny mechanizm "zastrzyku stresu" widać przy niebieskich liniach. Przewijanie mądrego tekstu nie relaksuje badanego; każdy ruch palcem zmusza prawą korę ciemieniową do gwałtownego reorientowania się w przestrzeni artykułu.

#### Płaty czołowe (Theta Fz, F4)

**Smart Scroll**: Po każdym przewinięciu (niebieska linia), płaty czołowe (zwłaszcza F4) generują górki fali Theta. Brak tu płynnego cyklu "zrozumienia".

**Brainrot Scroll**: W tym trybie widać aktywność "Wewnętrznego Krytyka". Tuż po czerwonych liniach scrolla na elektrodzie Fz pojawiają się punktowe, ostre wyskoki uwagi. Badany nie chłonie tego biernie – po każdym przewinięciu jego czoło aktywnie ocenia to, co właśnie wylosował mu algorytm.

#### Kora ruchowa (Beta C3)
**Smart Scroll**: Moment tuż po niebieskiej linii to wyraźny skok (rebound) fali Beta. Badany wkłada sporo intencjonalnej siły w przewijanie tekstów, co idealnie pasuje do faktu, że to zadanie go stresuje i traktuje je zadaniowo.

**Brainrot Scroll**: Przy szybkim przewijaniu głupotek linia Bety zrównuje się do postaci "szczotki", co oznacza, że prawa ręka weszła w tryb pełnego, napiętego automatyzmu. Palec pracuje tu niemal niezależnie od świadomości.

---
Ten badany to przykład na to, jak obronnie potrafi reagować układ nerwowy. Przewijanie ekranu nie służy mu do odkrywania nowości, ale działa jak "tarcza": odcina wzrok (skoki Alpha na O1), wywołuje stres przestrzenny (piki Theta na P4) i zmusza czoło do ciężkiej pracy analitycznej.